# FairScreen Baseline – EDA of the Resume–Job Matching Dataset

This notebook explores and cleans the Kaggle `resume_data.csv` dataset by Saugata Roy Arghya , where each row pairs one candidate's CV with one job and a `matched_score`, to confirm it is workable for a naive keyword-matching baseline that ranks CVs by how many job keywords they contain.

In [1]:
import pandas as pd

df = pd.read_csv("../data/resume_data.csv", encoding="utf-8-sig")
print(df.shape)

(9544, 35)


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9544 entries, 0 to 9543
Data columns (total 35 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   address                              784 non-null    str    
 1   career_objective                     4740 non-null   str    
 2   skills                               9488 non-null   str    
 3   educational_institution_name         9460 non-null   str    
 4   degree_names                         9460 non-null   str    
 5   passing_years                        9460 non-null   str    
 6   educational_results                  9460 non-null   str    
 7   result_types                         9460 non-null   str    
 8   major_field_of_studies               9460 non-null   str    
 9   professional_company_names           9460 non-null   str    
 10  company_urls                         9460 non-null   str    
 11  start_dates                          9460

In [4]:
df.iloc[0]

address                                                                              NaN
career_objective                       Big data analytics working and database wareho...
skills                                 ['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...
educational_institution_name           ['The Amity School of Engineering & Technology...
degree_names                                                                  ['B.Tech']
passing_years                                                                   ['2019']
educational_results                                                              ['N/A']
result_types                                                                      [None]
major_field_of_studies                                                   ['Electronics']
professional_company_names                                                 ['Coca-COla']
company_urls                                                                      [None]
start_dates          

### Findings from the first look

- **List-like text:** `skills` looks like a Python list (`['Big Data', 'Hadoop', ...]`) but is stored as a string. `related_skils_in_job` is a list inside a list (`[['Big Data']]`). These must be parsed before keywords can be counted.
- **Two kinds of missing values:** real blanks appear as `NaN`, but hidden blanks sit inside the text as `['N/A']` or `[None]` (e.g. `educational_results`, `locations`, `result_types`). `df.info()` counts these as non-null, so it understates how much data is missing.
- **Missing values come in blocks:** columns from the same CV section share the same non-null count: education and work history (9,460), extracurriculars (3,426), certifications (2,008), languages (700). When a CV lacks a section, all of that section's columns are empty together.
- **Duplicate column suspected:** in row 0, `responsibilities` and `responsibilities.1` contain the same text. This is checked across all rows in Section 2.
- **Hidden character in a column name:** column 28 appears as `job_position_name` with an invisible BOM character (`\ufeff`) in front of it, so it cannot be referenced by its normal name until fixed.
- **Early sign of a noisy score:** row 0 is a Big Data Analyst with an Electronics degree, scored **0.85** for a Senior Software Engineer job that has no `skills_required` listed.

In [6]:
same = (df["responsibilities"] == df["responsibilities.1"]).mean()
print(same)

1.0


In [7]:
job_cols = ["job_position_name", "educationaL_requirements", "experiencere_requirement",
            "age_requirement", "responsibilities.1", "skills_required"]
target = "matched_score"
cv_cols = [c for c in df.columns if c not in job_cols + [target, "responsibilities"]]
print(len(cv_cols), len(job_cols))

27 6


### Column map

Each row is **one CV paired with one job**, plus a score for how well they match. The 35 columns split into four groups:

| Group | Count | Columns | Role in this project |
|---|---|---|---|
| **Candidate (CV) side** | 27 | address, career_objective, skills, education (institution, degree, passing years, results, result types, major), work history (company names, URLs, start/end dates, related skills, positions, locations), extracurriculars (4), languages, proficiency levels, certifications (5) | Describes the applicant. `skills` and `related_skils_in_job` supply the CV keywords for the baseline. |
| **Job side** | 6 | job_position_name, educationaL_requirements, experiencere_requirement, age_requirement, responsibilities.1, skills_required | Describes the job. `job_position_name`, `skills_required` and `responsibilities.1` supply the job keywords. |
| **Target** | 1 | matched_score | The dataset's score (0–1) for how well the CV fits the job. Used to evaluate the baseline's ranking, never as an input. |
| **Duplicate (to drop)** | 1 | responsibilities | Looks like a CV field but is 100% identical to the job's `responsibilities.1`, so it carries no candidate information. |

Note: several column names contain typos from the source (`educationaL_requirements`, `experiencere_requirement`, `related_skils_in_job`). They are kept as-is for now and renamed during cleaning.